# Validação externa — Georgia 12-lead (PhysioNet/CinC 2020) | Ablation #1 v3 INT8

Avalia o **modelo embarcado final** (Ablation #1 v3: No-LOESS/NLM, split por paciente,
treino determinístico) no Georgia, terceira instituição ausente do treino, da
calibração e do teste. Nenhum parâmetro é ajustado no Georgia.

**Mudanças em relação à versão anterior (`eval_georgia_ablation_loess_nlm.ipynb`):**
1. Modelo, scaler e τ lidos de `Ablation_LOESS_NLM_v3_ok/` (SHA-256 do `.tflite` verificado).
2. τ = **τ do firmware** (grade INT8, critério A), e não o τ float32.
3. Butterworth = **mesma implementação do treino/firmware** (BLT com pre-warping, dois
   passes `sosfilt` com estado zerado), em vez de `scipy.sosfiltfilt` (que usa padding
   e condições iniciais diferentes nas bordas).
4. ACF em float64, idêntica a `extract_acf()` do treino.
5. Conversão de amplitude avaliada de duas formas, com diagnóstico contra o treino:
   - **A (principal): ganho do cabeçalho WFDB** — contagens ÷ ganho (unidades/mV),
     operação determinística igual para todos os sinais;
   - **B (sensibilidade): normalização por sinal** para std = 0,12 mV (procedimento anterior).
6. Métricas, bootstrap (n=1000, seed 42) e saídas no mesmo padrão das tabelas v3.

In [1]:
import os, glob, pickle, re, time, json, hashlib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import math as _math
from scipy.signal import sosfilt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

try:
    from ai_edge_litert.interpreter import Interpreter
    print('Usando ai_edge_litert.Interpreter')
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
    print(f'Usando tf.lite.Interpreter (TensorFlow {tf.__version__})')

# ── Modelo final (Ablation #1 v3) ───────────────────────────────────────
BASE_MODEL   = Path('/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3_ok')
TFLITE_PATH  = BASE_MODEL / 'ablation_loess_nlm_int8.tflite'
SCALER_PATH  = BASE_MODEL / 'ablation_loess_nlm_scaler.pkl'
THRESH_TFL   = BASE_MODEL / 'ecg_hybrid_v15_sem_loess_nlm_threshold_tflite.pkl'
METRICS_IN   = BASE_MODEL / 'metrics_final.json'
EXPECTED_SHA = 'a7582a6d1dd772754886b242c2c23a9c1fd36a7dbf7d261e4761e14b5e086b2e'

# ── Georgia ─────────────────────────────────────────────────────────────
GEORGIA_DIR  = Path('/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/raw/Georgia')
# Pasta com os cabeçalhos WFDB originais (E#####.hea). Ajustar se estiverem em outro local.
HEA_DIRS     = [GEORGIA_DIR,
                Path('/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/raw'),
                Path('/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/WFDB')]
ADC_GAIN_DEFAULT = 1000.0   # unidades por mV (padrão dos cabeçalhos do PhysioNet/CinC 2020) — usado só se nenhum .hea for encontrado
OUT_DIR      = Path('/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/eval_ablation_loess_nlm_v3')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Amostras do treino para diagnóstico de amplitude ────────────────────
TRAIN_DIRS = {'CSN': ['/mnt/windows/Doutorado/Datasets/Chinese_Lead_I_AFIB/',
                      '/mnt/windows/Doutorado/Datasets/Chinese_Lead_I_SR/'],
              'PTB': ['/mnt/windows/Doutorado/Datasets/PTB_XL_Lead_I_AFIB/',
                      '/mnt/windows/Doutorado/Datasets/PTB_XL_Lead_I_SR/']}
N_TRAIN_SAMPLE = 500        # sinais por diretório para o diagnóstico

FS, N_POINTS, N_ACF_LAGS = 500, 5000, 5000
TARGET_STD_TRAIN = 0.12     # mV — apenas para a variante B
N_BOOT, SEED = 1000, 42

for p in [TFLITE_PATH, SCALER_PATH, THRESH_TFL, GEORGIA_DIR]:
    assert p.exists(), f'Path obrigatório não encontrado: {p}'

sha = hashlib.sha256(open(TFLITE_PATH, 'rb').read()).hexdigest()
assert sha == EXPECTED_SHA, f'TFLite diferente do modelo v3 verificado: {sha}'
with open(SCALER_PATH, 'rb') as f: scaler = pickle.load(f)
with open(THRESH_TFL, 'rb') as f: thr = pickle.load(f)
TAU_FW = float(thr['tau_firmware'])
assert abs(TAU_FW - 0.8125) < 1e-9, f'tau_firmware inesperado: {TAU_FW}'
print(f'Modelo    : {TFLITE_PATH}\n  sha256  : {sha[:16]}... (confere com v3)')
print(f'Scaler    : n_features={scaler.n_features_in_}')
print(f'tau (firmware, grade INT8): {TAU_FW:.8f}')
print(f'Saída     : {OUT_DIR}')

Usando ai_edge_litert.Interpreter
Modelo    : /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3_ok/ablation_loess_nlm_int8.tflite
  sha256  : a7582a6d1dd77275... (confere com v3)
Scaler    : n_features=5000
tau (firmware, grade INT8): 0.81250000
Saída     : /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/eval_ablation_loess_nlm_v3


In [2]:
FN_RE = re.compile(r'^(E\d+)_(SR|AFIB)\.csv$', re.IGNORECASE)
records = []
for fp in sorted(GEORGIA_DIR.glob('*.csv')):
    m = FN_RE.match(fp.name)
    if m:
        records.append({'record_id': m.group(1), 'label_str': m.group(2).upper(),
                         'label_int': int(m.group(2).upper() == 'AFIB'),
                         'filepath': str(fp), 'filename': fp.name})
df = pd.DataFrame(records)
print(f'Registros: {len(df)}  SR={(df.label_int==0).sum()}  AFIB={(df.label_int==1).sum()}')
assert len(df) == 2069 and (df.label_int == 1).sum() == 496 and (df.label_int == 0).sum() == 1573
y_true = df['label_int'].values.astype(np.int64)

Registros: 2069  SR=1573  AFIB=496


## Leitura, ganho do cabeçalho WFDB e conversão para mV

In [3]:
GAIN_RE = re.compile(r'\s(-?\d+(?:\.\d+)?)(?:\(-?\d+\))?/mV')

def find_header(rec_id):
    for d in HEA_DIRS:
        p = d / f'{rec_id}.hea'
        if p.exists():
            return p
    return None

def lead_i_gain(hea_path):
    # Retorna o ganho (unidades/mV) da derivação I no cabeçalho WFDB.
    with open(hea_path) as f:
        lines = f.read().strip().splitlines()
    for ln in lines[1:]:
        toks = ln.split()
        if toks and toks[-1] == 'I':
            m = GAIN_RE.search(' ' + ln)
            if m:
                return float(m.group(1))
    return None

X_adc = np.zeros((len(df), N_POINTS), dtype=np.float64)
gains = np.full(len(df), np.nan)
for i, row in enumerate(df.itertuples()):
    arr = pd.read_csv(row.filepath, header=None).values.flatten().astype(np.float64)
    X_adc[i] = arr[:N_POINTS] if len(arr) >= N_POINTS else np.pad(arr, (0, N_POINTS - len(arr)))
    h = find_header(row.record_id)
    if h is not None:
        g = lead_i_gain(h)
        if g: gains[i] = g

n_hdr = int(np.isfinite(gains).sum())
if n_hdr == 0:
    print(f'⚠️  Nenhum .hea encontrado em {HEA_DIRS}.')
    print(f'    Usando ADC_GAIN_DEFAULT = {ADC_GAIN_DEFAULT} unidades/mV. CONFIRMAR no cabeçalho original.')
    gains[:] = ADC_GAIN_DEFAULT
    GAIN_SOURCE = 'default (nao verificado no .hea)'
else:
    print(f'Cabeçalhos lidos: {n_hdr}/{len(df)}   ganhos distintos: {sorted(set(gains[np.isfinite(gains)]))}')
    assert n_hdr == len(df), 'Há registros sem cabeçalho — verificar HEA_DIRS'
    GAIN_SOURCE = 'WFDB header (lead I)'

# Variante A — ganho do cabeçalho (principal)
X_mv_A = X_adc / gains[:, None]
# Variante B — normalização por sinal (procedimento anterior; análise de sensibilidade)
sd = X_adc.std(axis=1, keepdims=True); sd[sd == 0] = 1.0
X_mv_B = X_adc / sd * TARGET_STD_TRAIN
print(f'Fonte do ganho: {GAIN_SOURCE}')

⚠️  Nenhum .hea encontrado em [PosixPath('/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/raw/Georgia'), PosixPath('/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/raw'), PosixPath('/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/WFDB')].
    Usando ADC_GAIN_DEFAULT = 1000.0 unidades/mV. CONFIRMAR no cabeçalho original.
Fonte do ganho: default (nao verificado no .hea)


## Butterworth idêntico ao treino/firmware e diagnóstico de amplitude

In [4]:
def _butter_sos_blt(fc, btype, fs=500.0):
    omega_a = 2.0 * fs * _math.tan(_math.pi * fc / fs)
    k = (2.0 * fs) / omega_a; k2 = k * k; sqrt2k = _math.sqrt(2.0) * k
    a0 = k2 + sqrt2k + 1.0
    if btype == 'high':
        b0, b1, b2 = k2 / a0, -2.0 * k2 / a0, k2 / a0
    else:
        b0, b1, b2 = 1.0 / a0, 2.0 / a0, 1.0 / a0
    a1 = 2.0 * (1.0 - k2) / a0; a2 = (k2 - sqrt2k + 1.0) / a0
    return np.array([[b0, b1, b2, 1.0, a1, a2]], dtype=np.float64)

_SOS_HP = _butter_sos_blt(0.5, 'high'); _SOS_LP = _butter_sos_blt(40.0, 'low')

def butterworth_apply(signal):
    # Cópia exata do treino: HP->LP forward, reverse, HP->LP backward, reverse (estado zerado).
    s = signal.astype(np.float64).copy()
    def _one_pass(x): return sosfilt(_SOS_LP, sosfilt(_SOS_HP, x))
    s = _one_pass(s); s = s[::-1].copy(); s = _one_pass(s); s = s[::-1].copy()
    return s.astype(np.float32)

X_filt_A = np.stack([butterworth_apply(x) for x in X_mv_A])
X_filt_B = np.stack([butterworth_apply(x) for x in X_mv_B])

# Referência de amplitude do treino (amostra, já filtrada com o mesmo Butterworth)
rng_s = np.random.RandomState(SEED); ref = {}
for name, dirs in TRAIN_DIRS.items():
    feats = []
    for d in dirs:
        fl = sorted(glob.glob(os.path.join(d, '*.csv')))
        if not fl: continue
        for f in rng_s.choice(fl, min(N_TRAIN_SAMPLE, len(fl)), replace=False):
            s = pd.read_csv(f, header=None).values.flatten().astype(np.float32)[:N_POINTS]
            if len(s) < N_POINTS: s = np.pad(s, (0, N_POINTS - len(s)))
            feats.append(butterworth_apply(s))
    if feats: ref[name] = np.stack(feats)

def amp_stats(X):
    sd = X.std(axis=1); mx = np.abs(X).max(axis=1)
    return [np.median(sd), np.percentile(sd, 95), np.median(mx), np.percentile(mx, 95), np.percentile(mx, 99)]

rows = [(k, *amp_stats(v)) for k, v in ref.items()]
rows += [('Georgia A (ganho .hea)', *amp_stats(X_filt_A)), ('Georgia B (std/sinal)', *amp_stats(X_filt_B))]
amp_df = pd.DataFrame(rows, columns=['Conjunto', 'std p50', 'std p95', '|max| p50', '|max| p95', '|max| p99'])
print('Amplitude após Butterworth (mV):')
print(amp_df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
amp_df.to_csv(OUT_DIR / 'georgia_amplitude_diagnostics.csv', index=False)

Amplitude após Butterworth (mV):
              Conjunto  std p50  std p95  |max| p50  |max| p95  |max| p99
                   CSN    0.098    0.185      0.579      1.147      1.407
                   PTB    0.118    0.240      0.697      1.380      1.746
Georgia A (ganho .hea)    0.109    0.198      0.664      1.235      1.577
 Georgia B (std/sinal)    0.114    0.118      0.685      0.834      0.898


## ACF (idêntica ao treino) e inferência LiteRT INT8

In [5]:
def extract_acf(signal):
    s = signal.flatten().astype(np.float64); n = len(s)
    s = s - s.mean(); r0 = np.dot(s, s)
    if r0 < 1e-10: return np.zeros(n, dtype=np.float32)
    r = np.correlate(s, s, mode='full')[n - 1:]
    return (r / r0).astype(np.float32)

interp = Interpreter(model_path=str(TFLITE_PATH)); interp.allocate_tensors()
ind = interp.get_input_details(); outd = interp.get_output_details()
i_raw = next(i for i, d in enumerate(ind) if 'raw' in d['name'].lower())
i_acf = next(i for i, d in enumerate(ind) if 'acf' in d['name'].lower())
rs, rz = ind[i_raw]['quantization']; as_, az = ind[i_acf]['quantization']; os_, oz = outd[0]['quantization']
q = lambda x, s, z: np.clip(np.round(x / s + z), -128, 127).astype(np.int8)
print(f'input_raw scale={rs:.6f} zp={rz} | input_acf scale={as_:.6f} zp={az} | out scale={os_:.6f} zp={oz}')
Q_THR = int(round(TAU_FW / os_ + oz)); print(f'Limiar inteiro Q = {Q_THR}')

def infer(X_filt):
    acf = scaler.transform(np.stack([extract_acf(x) for x in X_filt])).astype(np.float32)
    out = np.zeros(len(X_filt), dtype=np.int32); sat = 0
    for i in range(len(X_filt)):
        rq = q(X_filt[i], rs, rz); sat += int(np.sum((rq == 127) | (rq == -128)))
        interp.set_tensor(ind[i_raw]['index'], rq.reshape(ind[i_raw]['shape']))
        interp.set_tensor(ind[i_acf]['index'], q(acf[i], as_, az).reshape(ind[i_acf]['shape']))
        interp.invoke()
        out[i] = int(interp.get_tensor(outd[0]['index']).flatten()[0])
    p = (out - oz) * os_
    print(f'  amostras saturadas na entrada raw: {sat / X_filt.size * 100:.3f}%')
    return out, p

print('Variante A (ganho do cabeçalho):'); qA, pA = infer(X_filt_A)
print('Variante B (std por sinal):');       qB, pB = infer(X_filt_B)

input_raw scale=0.009042 zp=0 | input_acf scale=0.107612 zp=-30 | out scale=0.003906 zp=-128
Limiar inteiro Q = 80
Variante A (ganho do cabeçalho):


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


  amostras saturadas na entrada raw: 0.073%
Variante B (std por sinal):


  amostras saturadas na entrada raw: 0.000%


## Métricas (padrão v3) — bootstrap n=1000, seed 42

In [6]:
KEYS = ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1-Score', 'Accuracy']
def compute(p, y, t):
    yh = (p >= t).astype(int); tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return {'AUC': roc_auc_score(y, p), 'Sensitivity': tp/(tp+fn), 'Specificity': tn/(tn+fp),
            'PPV': tp/(tp+fp) if tp+fp else np.nan, 'NPV': tn/(tn+fn) if tn+fn else np.nan,
            'F1-Score': 2*tp/(2*tp+fp+fn), 'Accuracy': (tp+tn)/len(y),
            'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)}
def boot(p, y, t):
    rng = np.random.RandomState(SEED); b = {k: [] for k in KEYS}
    for _ in range(N_BOOT):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        m = compute(p[i], y[i], t)
        for k in KEYS: b[k].append(m[k])
    return {k: [float(np.nanpercentile(b[k], 2.5)), float(np.nanpercentile(b[k], 97.5))] for k in KEYS}

res = {}
for tag, p in [('A_header_gain', pA), ('B_per_signal_std', pB)]:
    pt, ci = compute(p, y_true, TAU_FW), boot(p, y_true, TAU_FW); res[tag] = (pt, ci)
    print(f'\n=== Georgia | {tag} | tau={TAU_FW:.6f} | n={len(y_true)} ===')
    print(f"  TP={pt['TP']}  FN={pt['FN']}  FP={pt['FP']}  TN={pt['TN']}")
    for k in KEYS: print(f'  {k:<12} & {pt[k]:.4f} & [{ci[k][0]:.4f}, {ci[k][1]:.4f}] \\\\')


=== Georgia | A_header_gain | tau=0.812500 | n=2069 ===
  TP=388  FN=108  FP=9  TN=1564
  AUC          & 0.9560 & [0.9419, 0.9683] \\
  Sensitivity  & 0.7823 & [0.7450, 0.8217] \\
  Specificity  & 0.9943 & [0.9904, 0.9975] \\
  PPV          & 0.9773 & [0.9622, 0.9905] \\
  NPV          & 0.9354 & [0.9229, 0.9472] \\
  F1-Score     & 0.8690 & [0.8458, 0.8929] \\
  Accuracy     & 0.9435 & [0.9333, 0.9536] \\



=== Georgia | B_per_signal_std | tau=0.812500 | n=2069 ===
  TP=391  FN=105  FP=10  TN=1563
  AUC          & 0.9533 & [0.9392, 0.9664] \\
  Sensitivity  & 0.7883 & [0.7514, 0.8277] \\
  Specificity  & 0.9936 & [0.9891, 0.9975] \\
  PPV          & 0.9751 & [0.9581, 0.9901] \\
  NPV          & 0.9371 & [0.9253, 0.9485] \\
  F1-Score     & 0.8718 & [0.8484, 0.8954] \\
  Accuracy     & 0.9444 & [0.9348, 0.9541] \\


## Exportação e comparação com o teste in-domain (v3)

In [7]:
pd.DataFrame({'record_id': df.record_id, 'filename': df.filename, 'label_int': y_true,
              'int8_out_A': qA, 'p_int8_A': pA, 'pred_A': (pA >= TAU_FW).astype(int),
              'int8_out_B': qB, 'p_int8_B': pB, 'pred_B': (pB >= TAU_FW).astype(int)}
             ).to_csv(OUT_DIR / 'georgia_probs_int8.csv', index=False)

rec = {'evaluation': 'Georgia external validation - Ablation #1 v3 INT8',
       'timestamp': datetime.now().isoformat(), 'model_sha256': sha,
       'tau_int8_firmware': TAU_FW, 'tau_q': Q_THR, 'gain_source': GAIN_SOURCE,
       'butterworth': 'BLT pre-warped, 2-pass sosfilt with zeroed state (identical to training/firmware)',
       'n': int(len(y_true)), 'n_afib': int(y_true.sum()), 'n_sr': int((y_true == 0).sum()),
       'bootstrap': {'n': N_BOOT, 'seed': SEED, 'method': 'percentile'}}
for tag, (pt, ci) in res.items():
    rec[tag] = {'point': {k: float(pt[k]) for k in KEYS}, 'ci95': ci,
                'confusion': {k: pt[k] for k in ['TP', 'FN', 'FP', 'TN']}}
with open(OUT_DIR / 'georgia_metrics_v3.json', 'w') as f: json.dump(rec, f, indent=2)

fig, ax = plt.subplots(figsize=(6.5, 6))
for tag, p, col in [('A: ganho do cabeçalho', pA, '#185FA5'), ('B: std por sinal', pB, '#D85A30')]:
    fpr, tpr, _ = roc_curve(y_true, p); m = compute(p, y_true, TAU_FW)
    ax.plot(fpr, tpr, lw=2, color=col, label=f"{tag}  AUC={m['AUC']:.4f}")
    ax.plot(1 - m['Specificity'], m['Sensitivity'], 'o', color=col, ms=8)
ax.plot([0, 1], [0, 1], '--', color='gray', lw=0.8)
ax.set_xlabel('FPR (1 - Specificity)'); ax.set_ylabel('TPR (Sensitivity)')
ax.set_title(f'Georgia (n={len(y_true)}) — Ablation #1 v3 INT8 | tau={TAU_FW:.4f}', fontsize=10)
ax.legend(loc='lower right', fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / 'georgia_roc_int8_v3.png', dpi=150); plt.show()

if METRICS_IN.exists():
    ind_m = json.load(open(METRICS_IN))['int8_with_tau_firmware']
    print(f"\n  {'Métrica':<12}{'In-domain v3':>14}{'Georgia A':>12}{'Georgia B':>12}")
    for k in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'F1-Score']:
        print(f"  {k:<12}{ind_m[k]:>14.4f}{res['A_header_gain'][0][k]:>12.4f}{res['B_per_signal_std'][0][k]:>12.4f}")
print(f'\nSalvos em {OUT_DIR}: georgia_probs_int8.csv, georgia_metrics_v3.json, '
      'georgia_roc_int8_v3.png, georgia_amplitude_diagnostics.csv')


  Métrica       In-domain v3   Georgia A   Georgia B
  AUC                 0.9972      0.9560      0.9533
  Sensitivity         0.9618      0.7823      0.7883
  Specificity         0.9879      0.9943      0.9936
  PPV                 0.9133      0.9773      0.9751
  F1-Score            0.9369      0.8690      0.8718

Salvos em /home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Georgia/eval_ablation_loess_nlm_v3: georgia_probs_int8.csv, georgia_metrics_v3.json, georgia_roc_int8_v3.png, georgia_amplitude_diagnostics.csv


/tmp/ipykernel_598510/1968313296.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT_DIR / 'georgia_roc_int8_v3.png', dpi=150); plt.show()
